<a href="https://colab.research.google.com/github/busb1992/felnezek_data_pipelines/blob/main/google_forms_sheet_filter_colab_MORPHED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Forms / Google Sheets szűrő notebook

Ez a notebook:
1. belép Google Sheets-be Colab alatt,
2. felolvassa a megadott Google Sheet megadott munkalapját,
3. létrehoz egy új Google Sheetet,
4. az eredeti munkalapot átmásolja `raw data` néven,
5. létrehoz egy `filtered data` munkalapot,
6. a user konfiguráció alapján szűri az adatokat,
7. email címenként megtartja a legfrissebb sort az időbélyeg alapján.
8. létrehoz egy `morphed data` munkalapot a `filtered data` alapján, egyszerűsített jelentkezési kóddal.


In [ ]:
# Colab auth + library setup
!pip -q install gspread pandas gspread-dataframe

from google.colab import auth
import google.auth
import gspread
from gspread_dataframe import set_with_dataframe
import pandas as pd

auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

print("Google Sheets auth OK")

Google Sheets auth OK


## 1) User konfiguráció

Itt kell átírni a változókat.

- `SOURCE_SPREADSHEET_URL_OR_ID`: lehet teljes Google Sheet link vagy csak sheet ID.
- `SOURCE_WORKSHEET_NAME`: ezt a munkalapot olvassa be.
- `FILTER_COLUMN`: ebben az oszlopban keresi a session stringet.
- `WHICH_SESSION`: erre a stringre szűr.
- `TIMESTAMP_COLUMN`: időbélyeg oszlop, Google Forms esetén gyakran `Timestamp` vagy `Időbélyeg`.
- `EMAIL_COLUMN`: email cím oszlop, Google Forms esetén gyakran `Email Address` vagy `E-mail cím`.

In [ ]:
# =========================
# USER CONFIG - EZT SZERKESZD
# =========================

SOURCE_SPREADSHEET_URL_ORA_ID = "https://docs.google.com/spreadsheets/d/1vjYQrMDRROJOvZ247lSNmG00AM7E-q0EAAJh5xovNTY/edit?gid=1115506620#gid=1115506620"
SOURCE_WORKSHEET_NAME = "Munkalap1"   # melyik munkalapról olvasson

OUTPUT_SPREADSHEET_NAME = "mind1"

FILTER_COLUMN = "BUDAPEST - pontos dátumok\nTalálkozási hely: Selah Coffee House"      # ebben az oszlopban kell keresni

# Fontos: a Google Forms válaszban így szerepel: "június 25. csütörtök - 14:00"
# Ha azt írod, hogy "14:00, június 25. csütörtök", a SMART_SESSION_MATCH akkor is megtalálja.
WHICH_SESSION = "péntek - 14:00"

TIMESTAMP_COLUMN = "Időbélyeg"       # pl. Timestamp vagy Időbélyeg
EMAIL_COLUMN = "E-mail-cím"          # ebben a mintában ez a kitöltők email címe

CASE_INSENSITIVE_MATCH = True        # kis/nagybetű ne számítson
CONTAINS_MATCH = True                # Forms több választ egy cellába tesz, ezért itt általában True kell
SMART_SESSION_MATCH = True           # True: kezeli a "14:00, június 25" vs "június 25 - 14:00" sorrendi eltérést is


LEADERS = [
    "Toth Janos",
    "Hajdu Ildiko",
    "Hegyvári Pál",
    "Suba Lukács",
    "László Simon",
    "Varga Szabolcs",
    "Dudas Adrienn",
    "Pap Ferenc",
    "Nemes Dávid",
    "Bús Bálint",
    "Vigh Roland",
    "Ifrim Dávid",
    "Szász Melinda",
    "Török Jona",
    "Erdélyi Áron",
    "Kovács Hanga",
]

MIN_GROUP_SIZE = 3
PREFERRED_MAX_GROUP_SIZE = 10
HARD_MAX_GROUP_SIZE = 20

## 2) Helper függvények

In [ ]:
import re
import unicodedata


def extract_spreadsheet_id(url_or_id: str) -> str:
    """Google Sheet URL-ből kiszedi az ID-t; ha már ID-t kap, azt adja vissza."""
    text = url_or_id.strip()
    if "/spreadsheets/d/" in text:
        return text.split("/spreadsheets/d/")[1].split("/")[0]
    return text


def normalize_colname(name: str) -> str:
    """Oszlopnevek összehasonlításához: whitespace-ek egységesítése."""
    return re.sub(r"\s+", " ", str(name).strip()).lower()


def normalize_text(value: str, case_insensitive: bool = True) -> str:
    """Cellaszöveg normalizálása biztonságos szűréshez."""
    text = "" if value is None else str(value)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower() if case_insensitive else text


def session_tokens(value: str, case_insensitive: bool = True) -> list[str]:
    """A session stringet darabokra bontja."""
    text = normalize_text(value, case_insensitive=case_insensitive)
    tokens = [part.strip() for part in re.split(r"[,;\-|]+", text) if part.strip()]
    return tokens


def smart_session_candidates(value: str, case_insensitive: bool = True) -> list[str]:
    """
    Kezeli ezt az eltérést:
    user config:  "14:00, június 25. csütörtök"
    Forms cella:  "június 25. csütörtök - 14:00"
    """
    text = normalize_text(value, case_insensitive=case_insensitive)
    times = re.findall(r"\b\d{1,2}:\d{2}\b", text)
    candidates = [] if times else [text]

    for time_value in times:
        date_part = text.replace(time_value, " ")
        date_part = re.sub(r"[,;\-|]+", " ", date_part)
        date_part = re.sub(r"\s+", " ", date_part).strip()
        if date_part:
            # A Google Forms cellában az egyes opciók így vannak: dátum - idő
            # Direkt nem használjuk az eredeti "14:00, dátum" formát, mert az két szomszédos opció határán hamisan matchelhet.
            candidates.append(f"{date_part} - {time_value}")
            candidates.append(f"{time_value} - {date_part}")
            candidates.append(f"{date_part} {time_value}")
            candidates.append(f"{time_value} {date_part}")

    # sorrendtartó dedupe
    seen = set()
    unique_candidates = []
    for candidate in candidates:
        if candidate and candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)
    return unique_candidates


def resolve_column(df: pd.DataFrame, requested_col: str) -> str:
    """Megkeresi az oszlopot pontosan vagy normalizálva."""
    if requested_col in df.columns:
        return requested_col

    requested_norm = normalize_colname(requested_col)
    matches = [col for col in df.columns if normalize_colname(col) == requested_norm]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"Nem találom ezt az oszlopot: '{requested_col}'.\n"
        f"Elérhető oszlopok: {list(df.columns)}"
    )


def build_session_mask(
    series: pd.Series,
    session_value: str,
    case_insensitive: bool = True,
    contains_match: bool = True,
    smart_session_match: bool = True,
) -> pd.Series:
    normalized_series = series.map(lambda x: normalize_text(x, case_insensitive=case_insensitive))
    normalized_session = normalize_text(session_value, case_insensitive=case_insensitive)

    if smart_session_match:
        candidates = smart_session_candidates(session_value, case_insensitive=case_insensitive)
        mask = pd.Series(False, index=series.index)
        for candidate in candidates:
            mask |= normalized_series.str.contains(candidate, na=False, regex=False)
        return mask

    if contains_match:
        return normalized_series.str.contains(normalized_session, na=False, regex=False)

    return normalized_series.eq(normalized_session)


def filter_dataframe(
    df: pd.DataFrame,
    filter_column: str,
    session_value: str,
    timestamp_column: str,
    email_column: str,
    case_insensitive: bool = True,
    contains_match: bool = True,
    smart_session_match: bool = True,
) -> pd.DataFrame:
    """Session alapján szűr, majd email címenként megtartja a legfrissebb sort."""
    filter_col = resolve_column(df, filter_column)
    timestamp_col = resolve_column(df, timestamp_column)
    email_col = resolve_column(df, email_column)

    work_df = df.copy()

    # Üres email sorok kidobása, mert ezekre nem lehet megbízhatóan deduplikálni
    work_df[email_col] = work_df[email_col].astype(str).str.strip()
    work_df = work_df[work_df[email_col].ne("") & work_df[email_col].str.lower().ne("nan")]

    mask = build_session_mask(
        work_df[filter_col],
        session_value=session_value,
        case_insensitive=case_insensitive,
        contains_match=contains_match,
        smart_session_match=smart_session_match,
    )

    filtered = work_df[mask].copy()

    filtered["__parsed_timestamp"] = pd.to_datetime(
        filtered[timestamp_col],
        errors="coerce",
        dayfirst=False,
        utc=False,
    )

    filtered = filtered.sort_values("__parsed_timestamp", ascending=True, na_position="first")
    latest = filtered.drop_duplicates(subset=[email_col], keep="last").copy()
    latest = latest.drop(columns=["__parsed_timestamp"])

    return latest.reset_index(drop=True)


def create_filtered_data_sheet(df):
    """
    1. Emailenként megtartja a legfrissebb beküldést.
    2. Csak ezután szűri a kiválasztott session alapján.
    """

    df = df.copy()

    # -------------------------------------------------
    # Timestamp rendezése
    # -------------------------------------------------
    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )

    # -------------------------------------------------
    # Legfrissebb válasz emailenként
    # -------------------------------------------------
    latest_df = (
        df.sort_values(TIMESTAMP_COLUMN)
          .drop_duplicates(
              subset=[EMAIL_COLUMN],
              keep="last"
          )
          .reset_index(drop=True)
    )

    # -------------------------------------------------
    # Session filter
    # -------------------------------------------------
    values = latest_df[FILTER_COLUMN].fillna("").astype(str)

    if CASE_INSENSITIVE_MATCH:
        values = values.str.lower().str.strip()
        session = WHICH_SESSION.lower().strip()
    else:
        values = values.str.strip()
        session = WHICH_SESSION.strip()

    if CONTAINS_MATCH:
        mask = values.str.contains(
            session,
            regex=False,
            na=False
        )
    else:
        mask = values == session

    filtered_df = latest_df.loc[mask].copy()

    print(f"Original rows: {len(df)}")
    print(f"Latest per email: {len(latest_df)}")
    print(f"Filtered rows: {len(filtered_df)}")

    return filtered_df


## 3) Forrás sheet beolvasása

In [ ]:
source_spreadsheet_id = extract_spreadsheet_id(SOURCE_SPREADSHEET_URL_ORA_ID)
source_spreadsheet = gc.open_by_key(source_spreadsheet_id)
source_worksheet = source_spreadsheet.worksheet(SOURCE_WORKSHEET_NAME)

records = source_worksheet.get_all_records()
df_raw = pd.DataFrame(records)

print(f"Beolvasott sorok száma: {len(df_raw)}")
print(f"Beolvasott oszlopok: {list(df_raw.columns)}")
df_raw.head()

Beolvasott sorok száma: 613
Beolvasott oszlopok: ['Időbélyeg', 'Név:', 'E-mail', 'Telefonszám:', 'Városod:', 'Gyülekezeted:', 'Mi az a terület, amiben látod magad? (Többet is jelölhetsz.)', 'Mennyire vagy tapasztalt utcai evangelizációban?', 'Beszélsz angolul? ', 'Megjegyzés - kivel szeretnél egy párban lenni utcai evangelizációnál? (Teljes nevet írj, pontos időponttal és várossal együtt. Fontos, hogy csak akkor tudunk párba tenni titeket, hogyha Ő is beírja a nevedet ide.)', 'Melyik városokban számíthatunk Rád? (Többet is jelölhetsz. A pontos idősávokról hamarosan tájékoztatunk, kérjük, figyeld az email fiókod!)', 'A F.f.e.sz. szolgálata felekezetközi, ezért szükséges egy egységes irányadó működés, az “Iránytű”. Az Iránytű elfogadása szükséges minden a FELNÉZEK szolgálatához csatlakozó önkéntes számára!\nKattintsd ide!', 'Az adataim megadásával kifejezetten hozzájárulok ahhoz, hogy a Felnézek Keresztény Evangelizációs Alapítvány a vonatkozó jogszabályoknak megfelelően kezelje, azokról

,Időbélyeg,Név:,E-mail,Telefonszám:,Városod:,Gyülekezeted:,"Mi az a terület, amiben látod magad? (Többet is jelölhetsz.)",Mennyire vagy tapasztalt utcai evangelizációban?,Beszélsz angolul?,"Megjegyzés - kivel szeretnél egy párban lenni utcai evangelizációnál? (Teljes nevet írj, pontos időponttal és várossal együtt. Fontos, hogy csak akkor tudunk párba tenni titeket, hogyha Ő is beírja a nevedet ide.)",...,"A F.f.e.sz. szolgálata felekezetközi, ezért szükséges egy egységes irányadó működés, az “Iránytű”. Az Iránytű elfogadása szükséges minden a FELNÉZEK szolgálatához csatlakozó önkéntes számára!\nKattintsd ide!","Az adataim megadásával kifejezetten hozzájárulok ahhoz, hogy a Felnézek Keresztény Evangelizációs Alapítvány a vonatkozó jogszabályoknak megfelelően kezelje, azokról másolatot készítsen és tartson. Kifejezetten hozzájárulok továbbá ahhoz, hogy az Alapítvány a rendelkezésére bocsátott személyes adataikat az Európai Parlament és Tanács (EU) 2016/679 rendelete alapján megfelelően kezelje, tárolja, feldolgozza és az arra jogosult illetékes hatóságok részére továbbítsa, azokról szükség szerint nyilatkozzon, az arra jogosult és illetékes hatóság részére tájékoztatást adjon. Az adatkezelésre vonatkozik az Alapítvány adatkezelési szabályzata is: Adatvédelmi Szabályzat",E-mail-cím,"KECSKEMÉT - pontos dátumok \n(Találkozási hely: Kecskeméti Pünkösdi Gyülekezet (Kecskemét, Bethlen krt. 16,)","PÉCS - pontos dátumok\n(Találkozási hely: Agapé Gyülekezet, Pécs, Rákóczi utca 50.)","EGER - pontos dátumok\nTalálkozási hely: Dávid Hősei Gyülekezet (Eger, Széchenyi u. 6.)",DEBRECEN - pontos dátumok\nTalálkozási hely: Nagytemplom,BUDAPEST - pontos dátumok\nTalálkozási hely: Selah Coffee House,1. oszlop,2. oszlop
0,2026.02.17. 20:34:42,Juhász Jeremi,Jeremi.juhasz@gmail.com,36300971710,Gyál,Bpa,"utcai evangelizáció (evangelista), utcai evang...",5,"Igen, társalgási szinten",,...,Elolvastam és elfogadom.,Elfogadom,,,,,,,,
1,2026.02.18. 12:56:40,Pogány András,andras.pogany5@gmail.com,+3630/855-7415,Százhalombatta,Hit Gyülekezete Százhalombatta,"imaszolgálat a fesztiválon, Bármiben szívesen ...",3,"Igen, társalgási szinten",Pajor Tamás,...,Elolvastam és elfogadom.,Elfogadom,,,,,,,,
2,2026.02.21. 19:09:07,Földesi Mátyás,matyasfoldesi@gmail.com,36707452008,8420 Zirc,"MPE , Zirci Bérea gyülekezet","utcai evangelizáció (evangelista), utcai evang...",4,"Igen, társalgási szinten",,...,Elolvastam és elfogadom.,Elfogadom,,,,,,,,
3,2026.02.22. 6:11:30,Kovácsné Dobi Anikó,aniko.kovacsnedobi@gmail.com,6703291022,Nagyhegyes,Dávid Sátora Gyülekezet,"imaháttér, Bármiben szívesen segítek (pakolás,...",1,Nem,,...,Elolvastam és elfogadom.,Elfogadom,,,,,,,,
4,2026.02.22. 11:00:29,Lakatos-Balog Nóra,lakatosbalognora559@gmail.com,6308603433,Hajdúhadház,Tüzes szívek gyülekezete,"Bármiben szívesen segítek (pakolás, rendezés, ...",1,Nem,Lakatos-Balog Karina,...,Elolvastam és elfogadom.,Elfogadom,,,,,,,,


## 4) Új Google Sheet létrehozása + raw data mentése

In [ ]:


# Meglévő spreadsheet megnyitása
output_spreadsheet = gc.open(OUTPUT_SPREADSHEET_NAME)

sheet_names = ["raw data", "filtered data", "morphed data", "groups"]

for sheet_name in sheet_names:
    try:
        ws = output_spreadsheet.worksheet(sheet_name)

        # Ha létezik → töröljük
        output_spreadsheet.del_worksheet(ws)
        print(f"{sheet_name} törölve")

    except gspread.exceptions.WorksheetNotFound:
        print(f"{sheet_name} nem létezett")


    print(f"{sheet_name} létrehozva")

    # Újra létrehozzuk
ws = output_spreadsheet.add_worksheet(
        title="raw data",
        rows="1000",
        cols="20"
    )

# Data feltöltése (példa – figyelj, hogy létezzen!)
raw_ws = output_spreadsheet.worksheet("raw data")
set_with_dataframe(raw_ws, df_raw)

print("Adatok feltöltve a meglévő spreadsheetbe")




raw data törölve
raw data létrehozva
filtered data törölve
filtered data létrehozva
morphed data törölve
morphed data létrehozva
groups törölve
groups létrehozva
Adatok feltöltve a meglévő spreadsheetbe


## 5) Filtered data létrehozása

In [ ]:

df_filtered = create_filtered_data_sheet(df=df_raw)

sheet_name = "filtered data"

try:
    filtered_ws = output_spreadsheet.worksheet(sheet_name)
    filtered_ws.clear()
except gspread.exceptions.WorksheetNotFound:
    filtered_ws = output_spreadsheet.add_worksheet(
        title=sheet_name,
        rows="1000",
        cols="20"
    )

set_with_dataframe(filtered_ws, df_filtered)

print(f"Filtered sorok száma: {len(df_filtered)}")
print("filtered data munkalap feltöltve")
print("Output Google Sheet:")
print(output_spreadsheet.url)

df_filtered.head()

# ========== READ FROM FILTERED DATA SHEET (not from memory) ==========
# This ensures that any manual edits to the filtered data sheet are preserved
filtered_ws_fresh = output_spreadsheet.worksheet("filtered data")
filtered_records = filtered_ws_fresh.get_all_records()
df_filtered = pd.DataFrame(filtered_records)
print(f"\n[REFRESHED] Filtered data read from sheet: {len(df_filtered)} rows")


Original rows: 613
Latest per email: 296
Filtered rows: 38
Filtered sorok száma: 38
filtered data munkalap feltöltve
Output Google Sheet:
https://docs.google.com/spreadsheets/d/15VeCW6bdiyOBn5Tibwaz4LCp_FL0z3aXhJJ90WN7ss4


,Időbélyeg,Név:,E-mail,Telefonszám:,Városod:,Gyülekezeted:,"Mi az a terület, amiben látod magad? (Többet is jelölhetsz.)",Mennyire vagy tapasztalt utcai evangelizációban?,Beszélsz angolul?,"Megjegyzés - kivel szeretnél egy párban lenni utcai evangelizációnál? (Teljes nevet írj, pontos időponttal és várossal együtt. Fontos, hogy csak akkor tudunk párba tenni titeket, hogyha Ő is beírja a nevedet ide.)",...,"A F.f.e.sz. szolgálata felekezetközi, ezért szükséges egy egységes irányadó működés, az “Iránytű”. Az Iránytű elfogadása szükséges minden a FELNÉZEK szolgálatához csatlakozó önkéntes számára!\nKattintsd ide!","Az adataim megadásával kifejezetten hozzájárulok ahhoz, hogy a Felnézek Keresztény Evangelizációs Alapítvány a vonatkozó jogszabályoknak megfelelően kezelje, azokról másolatot készítsen és tartson. Kifejezetten hozzájárulok továbbá ahhoz, hogy az Alapítvány a rendelkezésére bocsátott személyes adataikat az Európai Parlament és Tanács (EU) 2016/679 rendelete alapján megfelelően kezelje, tárolja, feldolgozza és az arra jogosult illetékes hatóságok részére továbbítsa, azokról szükség szerint nyilatkozzon, az arra jogosult és illetékes hatóság részére tájékoztatást adjon. Az adatkezelésre vonatkozik az Alapítvány adatkezelési szabályzata is: Adatvédelmi Szabályzat",E-mail-cím,"KECSKEMÉT - pontos dátumok \n(Találkozási hely: Kecskeméti Pünkösdi Gyülekezet (Kecskemét, Bethlen krt. 16,)","PÉCS - pontos dátumok\n(Találkozási hely: Agapé Gyülekezet, Pécs, Rákóczi utca 50.)","EGER - pontos dátumok\nTalálkozási hely: Dávid Hősei Gyülekezet (Eger, Széchenyi u. 6.)",DEBRECEN - pontos dátumok\nTalálkozási hely: Nagytemplom,BUDAPEST - pontos dátumok\nTalálkozási hely: Selah Coffee House,1. oszlop,2. oszlop
31,2026-05-12 12:30:46,Balogh Eszter,,+36 30 8513045,Tarján,Agapé Gyülekezet Pestszentlőrinc,"Bármiben szívesen segítek (pakolás, rendezés, ...",1,"Igen, társalgási szinten",,...,Elolvastam és elfogadom.,Elfogadom,eszter.balogh33@gmail.com,,,,,"június 26. péntek - 14:00, június 26. péntek -...",,
45,2026-05-14 09:34:27,Gutveiler Gerda,,6205357119,Paks,Szabker Tengelic,"utcai evangelizáció (evangelista), média, imah...",1,"Igen, evangelizálni is tudok angolul, szívesen...",,...,Elolvastam és elfogadom.,Elfogadom,gerdagutveiler221@gmail.com,,,,,"június 26. péntek - 14:00, június 26. péntek -...",,
54,2026-05-24 13:57:51,SZALAI SZILVIA,,36304121005,BUDAPEST,B18 BAPTISTA KÖZÖSSÉG,"utcai evangelizáció (evangelista), Bármiben sz...",2,"Igen, társalgási szinten",KÁKONYI KRISZTINA; HEGYI GÁBORNÉ MARIKA,...,Elolvastam és elfogadom.,Elfogadom,silviaszalai72@gmail.com,,,,,"június 22. hétfő - 14:00, június 22. hétfő - 1...",,
72,2026-06-02 09:06:31,Korsós Erzsébet,,6705155802,Dorog,Agapé Somogyi Attila,"utcai evangelizáció (evangelista), imaháttér",2,Nem,Dóra Balázs,...,Elolvastam és elfogadom.,Elfogadom,kkunne@gmail.com,,"június 4. csütörtök - 10:00, június 4. csütört...","június 11. csütörtök - 10:00, június 11. csütö...",július 4. szombat - 14:00,"június 26. péntek - 10:00, június 26. péntek -...",,
73,2026-06-02 09:54:00,Toth Eszter,,6703287838,Budapest,Pünkösdi,"utcai evangelizáció (evangelista), utcai evang...",3,"Igen, társalgási szinten",Lány,...,Elolvastam és elfogadom.,Elfogadom,esztitoth96@gmail.com,,,,,"június 22. hétfő - 10:00, június 22. hétfő - 1...",,


## 7) Morphed data létrehozása

A `filtered data` alapján egy rövidebb, használhatóbb nézetet készít.

Megmarad a név, létrejön az `evangelizacio_kod`, a hosszú megjegyzés oszlop neve `megjegyzes` lesz, és megmarad az angol nyelv oszlop.


In [ ]:
# =========================
# MORPHED DATA CONFIG
# =========================

# Ha a név oszlop nálad máshogy hívódik, ezt írd át.
# A helper lent megpróbál több gyakori név-oszlopot is megtalálni.
NAME_COLUMN = "Név:"

AREA_COLUMN = "Mi az a terület, amiben látod magad? (Többet is jelölhetsz.)"
EXPERIENCE_COLUMN = "Mennyire vagy tapasztalt utcai evangelizációban?"

PAIR_NOTE_COLUMN = "Megjegyzés - kivel szeretnél egy párban lenni utcai evangelizációnál? (Teljes nevet írj, pontos időponttal és várossal együtt. Fontos, hogy csak akkor tudunk párba tenni titeket, hogyha Ő is beírja a nevedet ide.)"
ENGLISH_COLUMN = "Beszélsz angolul?"


def resolve_first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str:
    """Az első létező oszlopot adja vissza a megadott jelöltek közül."""
    for candidate in candidates:
        try:
            return resolve_column(df, candidate)
        except ValueError:
            pass
    raise ValueError(
        "Nem találom egyik név oszlopot sem ezek közül: "
        f"{candidates}\nElérhető oszlopok: {list(df.columns)}"
    )


def normalize_for_role_match(value) -> str:
    """Szöveg normalizálása szerepkör kereséshez."""
    text = normalize_text(value, case_insensitive=True)
    return text


def extract_experience_number(value) -> str:
    """Kiszedi a tapasztalati szint számát. Példa: '5' vagy '5 - sok' -> '5'."""
    text = "" if value is None else str(value).strip()
    match = re.search(r"\d+", text)
    return match.group(0) if match else ""


def build_evangelism_code(area_value, experience_value) -> str:
    """
    Példák:
    - utcai evangelizáció (evangelista), tapasztalat 5 -> E5
    - utcai evangelizáció (tanítványozó), tapasztalat 3 -> T3
    - mindkettő, tapasztalat 2 -> ET2
    """
    area_text = normalize_for_role_match(area_value)
    exp_num = extract_experience_number(experience_value)

    code = ""
    if "utcai evangelizáció (evangelista)" in area_text:
        code += "E"
    if "utcai evangelizáció (tanítványozó)" in area_text:
        code += "T"

    if not code:
        return ""

    return f"{code}{exp_num}"


def create_morphed_dataframe(df_filtered: pd.DataFrame) -> pd.DataFrame:
    """Filtered data-ból elkészíti a morphed data táblát."""
    name_col = resolve_first_existing_column(
        df_filtered,
        [
            NAME_COLUMN
        ],
    )
    area_col = resolve_column(df_filtered, AREA_COLUMN)
    experience_col = resolve_column(df_filtered, EXPERIENCE_COLUMN)
    pair_note_col = resolve_column(df_filtered, PAIR_NOTE_COLUMN)
    english_col = resolve_column(df_filtered, ENGLISH_COLUMN)

    morphed = pd.DataFrame()
    morphed["nev"] = df_filtered[name_col]
    morphed["evangelizacio_kod"] = df_filtered.apply(
        lambda row: build_evangelism_code(row[area_col], row[experience_col]),
        axis=1,
    )
    morphed["megjegyzes"] = df_filtered[pair_note_col]
    morphed[english_col] = df_filtered[english_col]

    return morphed.reset_index(drop=True)


# ========== READ FROM FILTERED DATA SHEET (not from memory) ==========
# This ensures that any manual edits to the filtered data sheet are picked up
filtered_ws_fresh = output_spreadsheet.worksheet("filtered data")
filtered_records = filtered_ws_fresh.get_all_records()
df_filtered_fresh = pd.DataFrame(filtered_records)
print(f"[REFRESHED] Filtered data read from sheet: {len(df_filtered_fresh)} rows")

# Morphed data létrehozása a filtered data alapján (reading fresh from sheet)
df_morphed = create_morphed_dataframe(df_filtered_fresh)

# Ha már létezik ilyen munkalap ugyanebben a futásban, töröljük és újra létrehozzuk
try:
    old_ws = output_spreadsheet.worksheet("morphed data")
    output_spreadsheet.del_worksheet(old_ws)
except Exception:
    pass

morphed_ws = output_spreadsheet.add_worksheet(
    title="morphed data",
    rows=max(len(df_morphed) + 10, 100),
    cols=max(len(df_morphed.columns) + 5, 10),
)
set_with_dataframe(morphed_ws, df_morphed)

print(f"Morphed sorok száma: {len(df_morphed)}")
print("morphed data munkalap feltöltve")
print("Output Google Sheet:")
print(output_spreadsheet.url)

# ========== READ FROM MORPHED DATA SHEET (not from memory) ==========
# This ensures that any manual edits to the morphed data sheet are preserved
morphed_ws_fresh = output_spreadsheet.worksheet("morphed data")
morphed_records = morphed_ws_fresh.get_all_records()
df_morphed = pd.DataFrame(morphed_records)
print(f"\n[REFRESHED] Morphed data read from sheet: {len(df_morphed)} rows")


Morphed sorok száma: 38
morphed data munkalap feltöltve
Output Google Sheet:
https://docs.google.com/spreadsheets/d/15VeCW6bdiyOBn5Tibwaz4LCp_FL0z3aXhJJ90WN7ss4


,nev,evangelizacio_kod,megjegyzes,Beszélsz angolul?
0,Balogh Eszter,,,"Igen, társalgási szinten"
1,Gutveiler Gerda,E1,,"Igen, evangelizálni is tudok angolul, szívesen..."
2,SZALAI SZILVIA,E2,KÁKONYI KRISZTINA; HEGYI GÁBORNÉ MARIKA,"Igen, társalgási szinten"
3,Korsós Erzsébet,E2,Dóra Balázs,Nem
4,Toth Eszter,ET3,Lány,"Igen, társalgási szinten"


## 8) Create Groups

In [ ]:
# ========== GROUPS CREATION - READ FROM MORPHED DATA SHEET ==========
# This ensures that any manual edits to the morphed data sheet are picked up

morphed_ws_for_groups = output_spreadsheet.worksheet("morphed data")
morphed_records_for_groups = morphed_ws_for_groups.get_all_records()
df_morphed_for_groups = pd.DataFrame(morphed_records_for_groups)
print(f"[REFRESHED] Morphed data read from sheet for groups: {len(df_morphed_for_groups)} rows")

# Create groups dataframe from fresh morphed data
groups_df = create_groups_dataframe(df_morphed_for_groups)
write_groups_sheet(output_spreadsheet, groups_df)

print("Groups created from fresh morphed data sheet")
groups_df.head(30)


In [ ]:
import re
import unicodedata
import pandas as pd

GROUP_COLORS = ["Zold", "Kek", "Sarga", "Piros", "Narancs","Lila"]

NAME_COLUMN = "nev"
RANK_CODE_COLUMN = "evangelizacio_kod"
NOTE_COLUMN = "megjegyzes"




def normalize_name(value):
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value


def extract_rank_number(code):
    if pd.isna(code):
        return 0

    match = re.search(r"\d+", str(code))
    return int(match.group()) if match else 0


def find_mentioned_participants(note, participant_names):
    """
    Megnézi, hogy a megjegyzés mezőben szerepel-e bármelyik résztvevő neve.
    Ékezet nélkül és kis/nagybetű függetlenül működik.
    """
    normalized_note = normalize_name(note)
    found = []

    for normalized_name, original_name in participant_names.items():
        if normalized_name and normalized_name in normalized_note:
            found.append(normalized_name)

    return found


def build_pair_units(df):
    """
    Olyan kis egységeket hoz létre, akiket együtt kell tartani.
    Ha A megjegyzésében szerepel B neve, akkor A és B egy unitba kerül.
    Többszörös kapcsolatoknál láncolva is működik:
    A -> B, B -> C esetén A/B/C együtt marad.
    """
    df = df.copy().reset_index(drop=True)
    df["row_id"] = df.index
    df["normalized_name"] = df[NAME_COLUMN].apply(normalize_name)

    participant_names = {
        row["normalized_name"]: row[NAME_COLUMN]
        for _, row in df.iterrows()
    }

    parent = {i: i for i in df["row_id"]}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        root_a = find(a)
        root_b = find(b)
        if root_a != root_b:
            parent[root_b] = root_a

    name_to_row_id = {
        row["normalized_name"]: row["row_id"]
        for _, row in df.iterrows()
    }

    for _, row in df.iterrows():
        mentioned_names = find_mentioned_participants(
            row.get(NOTE_COLUMN, ""),
            participant_names
        )

        for mentioned_name in mentioned_names:
            if mentioned_name in name_to_row_id:
                union(row["row_id"], name_to_row_id[mentioned_name])

    units = {}

    for _, row in df.iterrows():
        root = find(row["row_id"])
        units.setdefault(root, []).append(row)

    unit_list = []

    for members in units.values():
        unit_rank_sum = sum(int(member["rank_number"]) for member in members)
        unit_list.append({
            "members": members,
            "size": len(members),
            "rank_sum": unit_rank_sum,
        })

    return unit_list


def create_groups_dataframe(morphed_df):
    df = morphed_df.copy()

    df["rank_number"] = df[RANK_CODE_COLUMN].apply(extract_rank_number)
    df["normalized_name"] = df[NAME_COLUMN].apply(normalize_name)

    leader_lookup = {normalize_name(name): name for name in LEADERS}

    leaders_df = df[df["normalized_name"].isin(leader_lookup.keys())].copy()
    participants_df = df[~df["normalized_name"].isin(leader_lookup.keys())].copy()

    if leaders_df.empty:
        raise ValueError("Nincs felismerhető leader a morphed data résztvevői között.")

    leaders_df = leaders_df.sort_values(NAME_COLUMN).reset_index(drop=True)

    groups = []

    for idx, (_, leader_row) in enumerate(leaders_df.iterrows()):
        groups.append({
            "group_id": idx + 1,
            "color": GROUP_COLORS[idx % len(GROUP_COLORS)],
            "leader": leader_row[NAME_COLUMN],
            "members": [leader_row],
            "rank_sum": int(leader_row["rank_number"]),
        })

    units = build_pair_units(participants_df)

    # Nagyobb / erősebb unitok előre, hogy ne a végén okozzanak gondot
    units = sorted(
        units,
        key=lambda u: (u["size"], u["rank_sum"]),
        reverse=True
    )

    for unit in units:
        available_groups = [
            g for g in groups
            if len(g["members"]) + unit["size"] <= HARD_MAX_GROUP_SIZE
        ]

        preferred_groups = [
            g for g in available_groups
            if len(g["members"]) + unit["size"] <= PREFERRED_MAX_GROUP_SIZE
        ]

        target_groups = preferred_groups if preferred_groups else available_groups

        if not target_groups:
            raise ValueError(
                f"Nincs elég hely ennek az együtt tartandó egységnek: "
                f"{[m[NAME_COLUMN] for m in unit['members']]}"
            )

        selected_group = sorted(
            target_groups,
            key=lambda g: (
                len(g["members"]),
                g["rank_sum"]
            )
        )[0]

        selected_group["members"].extend(unit["members"])
        selected_group["rank_sum"] += unit["rank_sum"]

    output_rows = []

    for group in groups:
        group_size = len(group["members"])
        group_rank_sum = group["rank_sum"]
        group_avg_rank = round(group_rank_sum / group_size, 2) if group_size else 0

        for position, member in enumerate(group["members"], start=1):
            output_rows.append({
                "group_id": group["group_id"],
                "color": group["color"],
                "leader": group["leader"],
                "person_name": member[NAME_COLUMN],
                "is_leader": position == 1,
                "evangelizacio_kod": member.get(RANK_CODE_COLUMN, ""),
                "rank_number": member.get("rank_number", 0),
                "megjegyzes": member.get(NOTE_COLUMN, ""),
                "group_size": group_size,
                "group_rank_sum": group_rank_sum,
                "group_avg_rank": group_avg_rank,
            })

    groups_df = pd.DataFrame(output_rows)

    too_small = (
        groups_df.groupby("group_id")["person_name"]
        .count()
        .reset_index(name="group_size")
    )

    too_small = too_small[too_small["group_size"] < MIN_GROUP_SIZE]

    if not too_small.empty:
        print("WARNING: vannak 4 főnél kisebb csoportok:")
        print(too_small)

    return groups_df

def create_groups_display_dataframe(groups_df):
    blocks = []

    for group_id, group in groups_df.groupby("group_id"):
        group = group.copy()

        color = group["color"].iloc[0]
        leader = group["leader"].iloc[0]

        rows = []
        rows.append([f"{color.upper()} GROUP", ""])
        rows.append(["Leader", leader])
        rows.append(["Participants", "Rank"])

        for _, row in group.iterrows():
            if row["is_leader"]:
                continue

            rows.append([
                row["person_name"],
                row["evangelizacio_kod"]
            ])

        rows.append(["", ""])
        rows.append(["", ""])
        rows.append(["", ""])

        blocks.append(rows)

    final_rows = []

    for i in range(0, len(blocks), 2):
        left_block = blocks[i]
        right_block = blocks[i + 1] if i + 1 < len(blocks) else []

        max_len = max(len(left_block), len(right_block))

        for j in range(max_len):
            left = left_block[j] if j < len(left_block) else ["", ""]
            right = right_block[j] if j < len(right_block) else ["", ""]

            final_rows.append(left + [""] + right)

    return pd.DataFrame(
        final_rows,
        columns=[
            "left_name",
            "left_rank",
            "",
            "right_name",
            "right_rank"
        ]
    )


def write_groups_sheet(output_spreadsheet, groups_df):
    worksheet_name = "groups"

    display_df = create_groups_display_dataframe(groups_df)

    try:
        old_ws = output_spreadsheet.worksheet(worksheet_name)
        output_spreadsheet.del_worksheet(old_ws)
    except Exception:
        pass

    ws = output_spreadsheet.add_worksheet(
        title=worksheet_name,
        rows=str(len(display_df) + 20),
        cols="5"
    )

    ws.update(
        [display_df.columns.tolist()] + display_df.fillna("").values.tolist()
    )

    return ws

groups_df = create_groups_dataframe(df_morphed)
write_groups_sheet(output_spreadsheet, groups_df)


,group_id,color,leader,person_name,is_leader,evangelizacio_kod,rank_number,megjegyzes,group_size,group_rank_sum,group_avg_rank
0,1,Zold,Pap Ferenc,Pap Ferenc,True,ET5,5,,19,59,3.11
1,1,Zold,Pap Ferenc,Szurkos Abel,False,ET5,5,Budapest jun.23 14:00 Horvath Gabor. Bp. Jun.2...,19,59,3.11
2,1,Zold,Pap Ferenc,Szilágyi Melinda Noa,False,,0,Szurkos Ábel Budapest 14:00 június 26.péntek,19,59,3.11
3,1,Zold,Pap Ferenc,Bangó Melinda,False,ET5,5,,19,59,3.11
4,1,Zold,Pap Ferenc,Johanics Tunde,False,E5,5,,19,59,3.11
5,1,Zold,Pap Ferenc,Jónás Margit,False,E5,5,,19,59,3.11
6,1,Zold,Pap Ferenc,Mária Frankó,False,ET5,5,,19,59,3.11
7,1,Zold,Pap Ferenc,Tamás Péter,False,E4,4,,19,59,3.11
8,1,Zold,Pap Ferenc,Károly Mónika,False,E4,4,Szombaton Dráma,19,59,3.11
9,1,Zold,Pap Ferenc,Toth Eszter,False,ET3,3,Lány,19,59,3.11


## 9) Image Generation

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import math
import os
import zipfile
from IPython.display import display as ipy_display, Image as IPyImage

# ----------------------------------------------------------
# CONFIG
# ----------------------------------------------------------

OUTPUT_DIR = "group_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CARD_WIDTH = 900

PADDING = 50
HEADER_HEIGHT = 90
LEADER_HEIGHT = 60
ROW_HEIGHT = 52

BACKGROUND     = "#FFFFFF"
HEADER_BG      = "#2E7D32"   # dark green
HEADER_TEXT    = "#FFFFFF"
LEADER_BG      = "#1565C0"   # dark blue
LEADER_TEXT    = "#FFFFFF"
ROW_BG_ODD     = "#F5F5F5"
ROW_BG_EVEN    = "#FFFFFF"
ROW_TEXT       = "#212121"
RANK_TEXT      = "#555555"
BORDER_COLOR   = "#BDBDBD"

# ----------------------------------------------------------
# Font loading — tries several common Colab/Linux paths
# ----------------------------------------------------------

def load_font(bold: bool, size: int) -> ImageFont.FreeTypeFont:
    candidates_bold = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
        "DejaVuSans-Bold.ttf",
    ]
    candidates_regular = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "DejaVuSans.ttf",
    ]
    paths = candidates_bold if bold else candidates_regular
    for path in paths:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

font_title    = load_font(bold=True,  size=38)
font_leader   = load_font(bold=True,  size=28)
font_name     = load_font(bold=False, size=26)
font_rank     = load_font(bold=True,  size=24)
font_header   = load_font(bold=True,  size=22)

# ----------------------------------------------------------
# Draw one group → one image file
# ----------------------------------------------------------

def draw_group_image(group_id, color, leader, participants_df):
    n_participants = len(participants_df)
    card_height = (
        PADDING
        + HEADER_HEIGHT
        + LEADER_HEIGHT
        + ROW_HEIGHT            # column header row
        + n_participants * ROW_HEIGHT
        + PADDING
    )

    img = Image.new("RGB", (CARD_WIDTH, card_height), BACKGROUND)
    draw = ImageDraw.Draw(img)

    y = PADDING

    # ── Header bar ──────────────────────────────────────────
    draw.rectangle([0, y, CARD_WIDTH, y + HEADER_HEIGHT], fill=HEADER_BG)
    draw.text(
        (PADDING, y + (HEADER_HEIGHT - 38) // 2),
        f"{color.upper()} GROUP",
        font=font_title,
        fill=HEADER_TEXT,
    )
    y += HEADER_HEIGHT

    # ── Leader bar ──────────────────────────────────────────
    draw.rectangle([0, y, CARD_WIDTH, y + LEADER_HEIGHT], fill=LEADER_BG)
    draw.text(
        (PADDING, y + (LEADER_HEIGHT - 28) // 2),
        f"Leader: {leader}",
        font=font_leader,
        fill=LEADER_TEXT,
    )
    y += LEADER_HEIGHT

    # ── Column headers ──────────────────────────────────────
    draw.rectangle([0, y, CARD_WIDTH, y + ROW_HEIGHT], fill="#E0E0E0")
    draw.line([0, y + ROW_HEIGHT, CARD_WIDTH, y + ROW_HEIGHT], fill=BORDER_COLOR, width=1)
    draw.text((PADDING, y + (ROW_HEIGHT - 22) // 2), "Résztvevő",  font=font_header, fill="#333333")
    draw.text((CARD_WIDTH - 160, y + (ROW_HEIGHT - 22) // 2), "Kód", font=font_header, fill="#333333")
    y += ROW_HEIGHT

    # ── Participant rows ─────────────────────────────────────
    for i, (_, row) in enumerate(participants_df.iterrows()):
        bg = ROW_BG_ODD if i % 2 == 0 else ROW_BG_EVEN
        draw.rectangle([0, y, CARD_WIDTH, y + ROW_HEIGHT], fill=bg)
        draw.line([0, y + ROW_HEIGHT, CARD_WIDTH, y + ROW_HEIGHT], fill=BORDER_COLOR, width=1)

        draw.text(
            (PADDING, y + (ROW_HEIGHT - 26) // 2),
            str(row["person_name"]),
            font=font_name,
            fill=ROW_TEXT,
        )
        rank_str = str(row.get("evangelizacio_kod", ""))
        draw.text(
            (CARD_WIDTH - 160, y + (ROW_HEIGHT - 24) // 2),
            rank_str,
            font=font_rank,
            fill=RANK_TEXT,
        )
        y += ROW_HEIGHT

    return img

# ----------------------------------------------------------
# Generate one image per group
# ----------------------------------------------------------

saved_files = []

for group_id, group in groups_df.groupby("group_id"):
    color  = group["color"].iloc[0]
    leader = group["leader"].iloc[0]
    participants = group[group["is_leader"] == False].copy()

    img = draw_group_image(group_id, color, leader, participants)

    filename = f"{OUTPUT_DIR}/group_{group_id:02d}_{color.lower()}.png"
    img.save(filename, dpi=(150, 150))
    saved_files.append(filename)
    print(f"Saved: {filename}  ({len(participants)} résztvevő)")

# ----------------------------------------------------------
# Zip all images for easy download
# ----------------------------------------------------------

zip_path = "group_images.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for f in saved_files:
        zf.write(f, os.path.basename(f))

print(f"\nAll images zipped → {zip_path}")

# ----------------------------------------------------------
# Preview each image inline
# ----------------------------------------------------------

for f in saved_files:
    ipy_display(IPyImage(filename=f, width=600))


Saved: group_assignments.png
